In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
import pandas as pd
import os

ROOT = Path.cwd()
os.chdir(ROOT)

# Change this when you want to switch the training run.
RUN_NAME = "gakumasu_ui_detector_yolo11n"
PREDICT_RUN_NAME = f"{RUN_NAME}_predict"
RUN_DIR = ROOT / "runs/detect" / RUN_NAME
PREDICT_DIR = ROOT / "runs/detect" / PREDICT_RUN_NAME
BEST_MODEL_PATH = RUN_DIR / "weights/best.pt"
RESULTS_CSV_PATH = RUN_DIR / "results.csv"

print(ROOT)

/home/shunya/python/YOLO-gakumasu-train


In [2]:
from ultralytics import YOLO

In [3]:
model = YOLO("yolo11n.pt")

In [4]:
import albumentations as A

custom_aug = [
    A.OneOf([
        A.MotionBlur(blur_limit=3, p=1.0),
        A.GaussianBlur(blur_limit=3, p=1.0),
    ], p=0.15),

    A.RandomBrightnessContrast(
        brightness_limit=0.12,
        contrast_limit=0.12,
        p=0.35,
    ),

    A.ImageCompression(
        quality_range=(65, 95),
        p=0.25,
    ),

    A.GaussNoise(
        std_range=(0.03, 0.10),
        p=0.15,
    ),

    A.RGBShift(
        r_shift_limit=8,
        g_shift_limit=8,
        b_shift_limit=8,
        p=0.10,
    ),
]

results = model.train(
    data="dataset/data.yaml",
    epochs=30,
    patience=100,
    batch=16,
    imgsz=640,
    workers=8,
    name=RUN_NAME,
    seed=0,
    deterministic=True,
    amp=True,
    plots=True,

    # Ultralytics標準DA：UIなので弱め
    hsv_h=0.005,
    hsv_s=0.25,
    hsv_v=0.20,
    translate=0.02,
    scale=0.05,
    flipud=0,
    fliplr=0,
    mosaic=0,
    mixup=0,
    cutmix=0,
    copy_paste=0,

    # Custom Albumentations
    augmentations=custom_aug,
)

New https://pypi.org/project/ultralytics/8.4.90 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.81 🚀 Python-3.12.3 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660, 6144MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[OneOf([
  MotionBlur(p=1.0, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(3, 3), direction_range=(-1.0, 1.0)),
  GaussianBlur(p=1.0, blur_limit=(0, 3), sigma_limit=(0.5, 3.0)),
], p=0.15), RandomBrightnessContrast(p=0.35, brightness_by_max=True, brightness_limit=(-0.12, 0.12), contrast_limit=(-0.12, 0.12), ensure_safe_range=False), ImageCompression(p=0.25, compression_type='jpeg', quality_range=(65, 95)), GaussNoise(p=0.15, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.03, 0.1)), RGBShift(p=0.1, approximation=1.0, b_shift_limit=(-8.0, 8.0), g_shift_limit=(-8.0, 8.0), noise_params={'noise_type': 'uniform', 'ranges': [(-0.03137254901960784, 0.03137254901960784), (-0

In [5]:
model = YOLO(BEST_MODEL_PATH)

results = model.predict(
    source=ROOT / "test_images",
    imgsz=640,
    conf=0.25,
    save=True,
    project=ROOT / "runs/detect",
    name=PREDICT_RUN_NAME,
)


image 1/20 /home/shunya/python/YOLO-gakumasu-train/test_images/test001.png: 640x320 13 skill_cards, 68.7ms
image 2/20 /home/shunya/python/YOLO-gakumasu-train/test_images/test002.png: 640x320 4 skill_cards, 13.1ms
image 3/20 /home/shunya/python/YOLO-gakumasu-train/test_images/test003.png: 640x320 22 skill_cards, 1 p_item, 2 support_cards, 16.5ms
image 4/20 /home/shunya/python/YOLO-gakumasu-train/test_images/test004.png: 640x320 6 skill_cards, 4 p_items, 1 idol_name, 1 vo_param, 1 da_param, 1 vi_param, 1 stamina, 20.0ms
image 5/20 /home/shunya/python/YOLO-gakumasu-train/test_images/test005.png: 640x320 12 skill_cards, 10.2ms
image 6/20 /home/shunya/python/YOLO-gakumasu-train/test_images/test006.png: 640x320 2 skill_cards, 9 p_items, 28.8ms
image 7/20 /home/shunya/python/YOLO-gakumasu-train/test_images/test007.png: 640x320 1 skill_card, 1 p_item, 1 idol_img, 1 idol_name, 1 vo_bonus, 1 da_bonus, 1 vi_bonus, 1 stamina, 6 support_cards, 5 memorys, 1 scenario_title, 1 difficulty_title, 1 kir

In [6]:
img = cv2.imread(str(PREDICT_DIR / "test014.jpg"))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis("off")
plt.show()

<Figure size 1000x600 with 1 Axes>

In [7]:
csv_path = RESULTS_CSV_PATH

df = pd.read_csv(csv_path)

plt.figure(figsize=(10, 6))

plt.plot(df["epoch"], df["train/box_loss"], label="Train Box")
plt.plot(df["epoch"], df["val/box_loss"], label="Val Box")

plt.plot(df["epoch"], df["train/cls_loss"], label="Train Cls")
plt.plot(df["epoch"], df["val/cls_loss"], label="Val Cls")

plt.plot(df["epoch"], df["train/dfl_loss"], label="Train DFL")
plt.plot(df["epoch"], df["val/dfl_loss"], label="Val DFL")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.grid(True)

plt.show()

<Figure size 1000x600 with 1 Axes>

In [8]:
plt.figure(figsize=(8,5))

plt.plot(df["epoch"], df["train/dfl_loss"], label="Train DFL")
plt.plot(df["epoch"], df["val/dfl_loss"], label="Val DFL")

plt.grid(True)
plt.legend()
plt.show()

<Figure size 800x500 with 1 Axes>

In [9]:
plt.figure(figsize=(10, 6))

plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation mAP")
plt.grid(True)
plt.legend()

plt.show()

<Figure size 1000x600 with 1 Axes>

In [10]:

best_model = YOLO(BEST_MODEL_PATH)
val_results = best_model.val(
    data=ROOT / "dataset/data.yaml",
    split="val",
    imgsz=640,
    plots=False,
    verbose=False,
)

class_metrics = pd.DataFrame(val_results.summary())
class_metrics = class_metrics.rename(
    columns={
        "Class": "class",
        "Images": "images",
        "Instances": "instances",
        "Box-P": "precision",
        "Box-R": "recall",
        "Box-F1": "f1",
        "mAP50": "mAP50",
        "mAP50-95": "mAP50-95",
    }
)
class_metrics = class_metrics.sort_values("mAP50-95", ascending=True).reset_index(drop=True)

display(class_metrics)

fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(class_metrics) * 0.35)), sharey=True)

metric_columns = ["precision", "recall", "mAP50", "mAP50-95"]
class_metrics.plot.barh(
    x="class",
    y=metric_columns,
    ax=axes[0],
    width=0.85,
)
axes[0].set_title("Validation Metrics by Class")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Class")
axes[0].set_xlim(0, 1.05)
axes[0].grid(axis="x", alpha=0.3)
axes[0].legend(loc="lower right")

axes[1].barh(class_metrics["class"], class_metrics["instances"], color="tab:gray")
axes[1].set_title("Validation Instances by Class")
axes[1].set_xlabel("Instances")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

Ultralytics 8.4.81 🚀 Python-3.12.3 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660, 6144MiB)
YOLO11n summary (fused): 101 layers, 2,586,637 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1469.7±1028.5 MB/s, size: 1193.9 KB)
val: Scanning /home/shunya/python/YOLO-gakumasu-train/dataset/labels/val.cache... 220 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 220/220 46.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 2.4it/s 5.9s0.3s
                   all        220       2607      0.992      0.997      0.995      0.912
Speed: 2.5ms preprocess, 6.8ms inference, 0.0ms loss, 3.5ms postprocess per image


,class,images,instances,precision,recall,f1,mAP50,mAP50-95
0,scenario_title,52,52,0.99084,1.00000,0.99540,0.99500,0.79501
1,vo_score,48,48,1.00000,0.98069,0.99025,0.99500,0.81223
2,kirameki,32,32,1.00000,0.97375,0.98670,0.99500,0.82251
3,vi_score,48,48,0.99330,1.00000,0.99664,0.99500,0.82478
4,da_score,48,48,1.00000,0.99173,0.99585,0.99500,0.84462
5,stamina,87,87,0.99876,1.00000,0.99938,0.99500,0.85690
6,star_param,14,14,0.96995,1.00000,0.98475,0.99500,0.86857
7,fan_count,76,76,0.99448,1.00000,0.99723,0.99500,0.87737
8,exam_score,48,48,0.99189,1.00000,0.99593,0.99500,0.88045
9,vo_param,87,87,0.99561,1.00000,0.99780,0.99500,0.91096


<Figure size 1400x805 with 2 Axes>

In [ ]:
# ONNXエクスポート
best_model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=12,
)

In [ ]:
# TensorRT エクスポート
best_model = YOLO(BEST_MODEL_PATH)

best_model.export(
    format="engine",
    imgsz=640,
    dynamic=False,
    simplify=True,
    half=False,
    int8=False,
    opset=12,
    nms=False,
)

WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.81 🚀 Python-3.12.3 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1660, 6144MiB)
YOLO11n summary (fused): 101 layers, 2,586,637 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/home/shunya/python/YOLO-gakumasu-train/runs/detect/gakumasu_ui_detector_yolo11n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 27, 8400) (5.2 MB)

ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.3s, saved as '/home/shunya/python/YOLO-gakumasu-train/runs/detect/gakumasu_ui_detector_yolo11n/weights/best.onnx' (10.1 MB)

TensorRT: starting export with TensorRT 11.1.0.106...
[07/09/2026-15:34:20] [TRT] [I] [MemUsageChange] Init CUDA: CPU +0, GPU +0, now: CPU 2461, GPU 1917 (MiB)
[07/09/2026-15:34:21] [TRT] [I] ------------------

PosixPath('/home/shunya/python/YOLO-gakumasu-train/runs/detect/gakumasu_ui_detector_yolo11n/weights/best.engine')

In [12]:
engine_model = YOLO("runs/detect/gakumasu_ui_detector_yolo11n/weights/best.engine")

In [13]:
import cv2
from matplotlib import pyplot as plt

# 1. TensorRTモデルで推論を実行
results = engine_model("test_images/test002.png")

# 2. 検出結果が描画された画像（BGRのNumpy配列）を取得
annotated_img = results[0].plot()

# 3. OpenCVのBGR形式から、matplotlib用のRGB形式に変換
annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

# 4. Colabのセル上に直接表示
plt.figure(figsize=(10, 10))  # 表示するサイズ（好みに合わせて調整してください）
plt.imshow(annotated_img_rgb)
plt.axis('off')               # 周りの目盛り（ピクセル数）を消す
plt.show()

Loading runs/detect/gakumasu_ui_detector_yolo11n/weights/best.engine for TensorRT inference...
[07/09/2026-15:35:41] [TRT] [I] Loaded engine size: 12 MiB
[07/09/2026-15:35:41] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +18, now: CPU 0, GPU 30 (MiB)

image 1/1 /home/shunya/python/YOLO-gakumasu-train/test_images/test002.png: 640x640 4 skill_cards, 18.5ms
Speed: 6.6ms preprocess, 18.5ms inference, 10.6ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 1000x1000 with 1 Axes>